# 🔍 XAI — explaining 3D segmentation predictions (both arms)

**📎 Required Kaggle Datasets:**
1. Output of **notebook 03** (Swin UNETR checkpoint `swinunetr_best.pt` + split + results)
2. *(Optional)* snapshot from **notebook 02** (`nnunet_snapshots/...`) for nnU-Net saliency
3. BraTS dataset for the images

**What we do**
1. Gradient-based **input saliency** for the Swin UNETR (repo's `src/xai.py`) + MONAI **GradCAM**
2. *(optional cell)* saliency through **nnU-Net's own predictor** (single pass, TTA off — documented)
3. Side-by-side overlays on the **best-Dice and worst-Dice validation cases** (failure analysis)
4. Honest method notes: what gradient saliency can and cannot tell us

> ⚠️ Method transparency (put this in your writeup): input-gradient saliency is a basic
> attribution method — fast and model-agnostic, but noisy. GradCAM is smoother (it aggregates
> gradients at feature maps). We show both; where they disagree, we say so instead of
> cherry-picking the prettier map. This kind of honesty is exactly what XAI reviewers look for.

In [ ]:
!pip install -q -U "monai[nibabel]>=1.3"
!git clone --depth 1 https://github.com/naveed1011/nnunet-medseg-benchmark.git /kaggle/working/repo
import os, sys, glob, json
sys.path.insert(0, "/kaggle/working/repo"); os.chdir("/kaggle/working/repo")
import numpy as np, pandas as pd, torch, nibabel as nib
from src.xai import saliency_torch_model, plot_overlay

## 1 · Load the trained Swin UNETR + pick analysis cases (best & worst Dice)

In [ ]:
from monai.networks.nets import SwinUNETR
device = torch.device("cuda")
model = SwinUNETR(img_size=(96,96,96), in_channels=4, out_channels=4, feature_size=24, use_v2=True).to(device)
ckpt_path = (glob.glob("/kaggle/input/**/swinunetr_best.pt", recursive=True) + glob.glob("/kaggle/working/swinunetr_best.pt"))[0]
ckpt = torch.load(ckpt_path, map_location=device); model.load_state_dict(ckpt["model"]); model.eval()
print("loaded", ckpt_path, "| val dice at save:", ckpt["val_dice"])

# split + per-case metrics
split = json.load(open((glob.glob("/kaggle/input/**/brats_subset_split.json", recursive=True) + glob.glob("/kaggle/working/**/brats_subset_split.json", recursive=True))[0]))
per_case = pd.read_csv((glob.glob("/kaggle/input/**/results/brats_swinunetr/per_case_metrics.csv", recursive=True) + glob.glob("/kaggle/working/results/brats_swinunetr/per_case_metrics.csv"))[0])
mean_dice = per_case.groupby("case")["dice"].mean().sort_values()
worst, best = mean_dice.index[0], mean_dice.index[-1]
print(f"worst case: {worst} (mean dice {mean_dice.iloc[0]:.3f}) | best: {best} ({mean_dice.iloc[-1]:.3f})")

## 2 · Rebuild the case loader (same preprocessing as training — critical!)

Saliency w.r.t. a *differently preprocessed* input is meaningless. We reuse the exact
normalization from notebook 03.

In [ ]:
MODS = ("flair", "t1", "t1ce", "t2")
_REMAP_LUT = np.array([0, 1, 2, 0, 3], dtype=np.int8)   # BraTS {0,1,2,4} -> {0,1,2,3}
cands = glob.glob("/kaggle/input/*brats*")
TD = (glob.glob(f"{cands[0]}/**/RSNA_ASNR_MICCAI_BraTS_2021_TrainingData", recursive=True) or [cands[0]])[0]
def load_case(cid):
    d = os.path.join(TD, split["id_map"][cid])
    X = np.stack([np.asarray(nib.load([f for f in glob.glob(os.path.join(d, f"*{m}.nii.gz")) if "seg" not in os.path.basename(f).lower()][0]).dataobj, dtype=np.float32) for m in MODS])
    y = _REMAP_LUT[np.clip(np.asarray(nib.load(glob.glob(os.path.join(d, "*seg.nii.gz"))[0]).dataobj).astype(np.int16), 0, 4)]
    return X, y
def normalize(x):
    fg = x[x > 0]
    if fg.size == 0: return x
    lo, hi = np.percentile(fg, [0.5, 99.5]); x = np.clip(x, lo, hi)
    fg = x[x > 0]
    return (x - fg.mean()) / (fg.std() + 1e-8)

## 3 · Saliency for one case, per tumor class (ET=3, ED=2, NCR=1)

The backward pass through the full 240×240×155 volume does not fit on a T4, so I explain a
**96³ window centred on the tumour** and say so in the figure title. The preprocessing inside
that window is identical to training — a saliency map computed on differently normalised
input would be meaningless.

In [ ]:
%%time
from monai.inferers import sliding_window_inference

CROP = 96

def tumor_crop(X, y, size=CROP):
    fg = np.argwhere(y > 0)
    c = fg.mean(0).astype(int) if len(fg) else np.array(X.shape[1:]) // 2
    starts = [int(np.clip(c[i] - size // 2, 0, X.shape[i + 1] - size)) for i in range(3)]
    sl = (slice(None),) + tuple(slice(s, s + size) for s in starts)
    return X[sl], y[starts[0]:starts[0]+size, starts[1]:starts[1]+size, starts[2]:starts[2]+size]

def analyze(cid, target_class, tag):
    X, y = load_case(cid)
    Xn = np.stack([normalize(c) for c in X]).astype(np.float32)
    Xc, yc = tumor_crop(Xn, y)
    xt = torch.from_numpy(Xc[None]).to(device)
    with torch.no_grad():
        logits = sliding_window_inference(xt, (CROP, CROP, CROP), 1, model, overlap=0.5)
        pred = torch.argmax(torch.softmax(logits, 1), 1)[0].cpu().numpy()
    sal = saliency_torch_model(model, xt, target_class=target_class)   # (CROP, CROP, CROP)
    outp = f"/kaggle/working/figures/xai_{tag}_c{target_class}.png"
    plot_overlay(Xc[0], pred, yc.astype(np.uint8), sal, axis=2,
                 title=f"{cid} — class {target_class} — gradient saliency ({tag}, {CROP}³ crop)",
                 out_path=outp)
    return outp, pred, sal

class_names = {1: "NCR/NET", 2: "Edema", 3: "Enhancing tumor"}
for cid, tag in [(best, "best"), (worst, "worst")]:
    for c in (2, 3):
        p, pred, sal = analyze(cid, c, tag)
        print(f"{tag} case {cid} class {class_names[c]} -> {p}")

## 4 · MONAI GradCAM (smoother attribution — cross-check)

GradCAM aggregates gradients at a target feature layer instead of the raw input, so maps are
less noisy. Agreement between the two methods increases confidence in the explanation.

In [ ]:
from monai.visualize import GradCAM
layer = "layers1"  # SwinUNETR encoder stage; try layers2/layers3 for coarser maps
try:
    cam = GradCAM(nn_module=model, target_layers=f"{layer}.0.blocks.0.attn")
    X, y = load_case(worst)
    Xn = np.stack([normalize(c) for c in X]).astype(np.float32)
    Xc, yc = tumor_crop(Xn, y)          # same 96^3 window as the gradient saliency
    xt = torch.from_numpy(Xc[None]).to(device)
    with torch.no_grad():
        logits = sliding_window_inference(xt, (CROP, CROP, CROP), 1, model, overlap=0.5)
        pred = torch.argmax(torch.softmax(logits, 1), 1)[0].cpu().numpy()
    for c in (2, 3):
        res = cam(x=xt, class_idx=torch.tensor([c]).to(device))
        heat = res[0].mean(0).detach().cpu().numpy()
        if heat.shape != Xc.shape[1:]:   # GradCAM may return a coarser map
            heat = torch.nn.functional.interpolate(
                torch.from_numpy(heat)[None, None], size=Xc.shape[1:],
                mode="trilinear", align_corners=False)[0, 0].numpy()
        heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
        plot_overlay(Xc[0], pred, yc.astype(np.uint8), heat, axis=2,
                     title=f"{worst} — class {c} — MONAI GradCAM ({layer}, {CROP}³ crop)",
                     out_path=f"/kaggle/working/figures/gradcam_worst_c{c}.png")
    print("✅ GradCAM figures saved")
except Exception as e:
    print("GradCAM target layer name differs in this MONAI version — introspect with:")
    print([n for n, _ in model.named_modules() if "blocks" in n][:12])
    print("error was:", e)

## 5 · (Optional) nnU-Net saliency via its own predictor

If notebook 02's **snapshot** (`nnunet_snapshots/Dataset701_BraTS21/...`) is attached, we can
compute saliency through nnU-Net's preprocessing + network — the repo's `nnunet_case_saliency`
handles the plumbing (single centered pass, TTA off, resampled back to original geometry).

> This cell needs the full nnU-Net env (env vars + preprocessed plans). If the snapshot lacks
> `plans.json`/dataset metadata, run notebook 02's setup cells first (no retraining needed).

In [ ]:
# OPTIONAL — uncomment when the nnU-Net snapshot is attached
# !pip install -q -U nnunetv2
# import os
# os.environ["nnUNet_raw"]          = "/kaggle/working/nnunet_env/raw"
# os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnunet_env/pre"
# os.environ["nnUNet_results"]      = "/kaggle/working/nnunet_env/res"
# # restore snapshot into $nnUNet_results, re-run plan_and_preprocess for Dataset701 if needed, then:
# from src.xai import nnunet_case_saliency, plot_overlay
# model_folder = "/kaggle/working/nnunet_env/res/Dataset701_BraTS21/nnUNetTrainer_100epochs__nnUNetPlans__3d_fullres/fold_0"
# cid = worst
# d = os.path.join(TD, split["id_map"][cid])
# files = [[f for f in glob.glob(os.path.join(d, f"*{m}.nii.gz")) if "seg" not in os.path.basename(f).lower()][0] for m in MODS]
# out = nnunet_case_saliency(model_folder, files, target_class=3)
# img0 = np.asarray(nib.load(files[0]).dataobj, dtype=np.float32)
# gt = _REMAP_LUT[np.clip(np.asarray(nib.load(glob.glob(os.path.join(d, "*seg.nii.gz"))[0]).dataobj).astype(np.int16), 0, 4)]
# plot_overlay(img0, out["prediction"], gt, out["saliency_orig"], axis=2,
#              title=f"nnU-Net saliency — {cid} — ET", out_path=f"/kaggle/working/figures/nnunet_saliency_{cid}.png")

## 6 · Writing up the XAI analysis (the part professors actually read)

For each figure, answer in 2–3 sentences:
1. **Does the model look where a radiologist would?** (saliency on tumor vs healthy tissue)
2. **Failure modes:** on the worst case, is saliency diffuse (uncertain), misplaced (confusion
   with vessels/artifacts), or correct-but-undersegmented (threshold/sensitivity issue)?
3. **Method caveat:** gradient saliency is a first-order explanation — noisy, no guarantees.
   State it. If maps disagree between methods, report the disagreement.

Suggested repo layout for results:
```
docs/figures/xai_best_c3.png
docs/figures/xai_worst_c2.png
docs/figures/gradcam_worst_c3.png
```